# Thực nghiệm mô hình học máy dự đoán nguy cơ bỏ học

Notebook huấn luyện và đánh giá hai thuật toán học máy trên hai nguồn dữ liệu độc lập:

- Logistic Regression được dùng làm mô hình cơ sở, có ưu điểm dễ giải thích.
- LightGBM được dùng làm mô hình nâng cao, có khả năng học các quan hệ phi tuyến.

Mỗi nguồn sử dụng đúng các tập huấn luyện, xác thực và kiểm thử đã tạo trong `outputs/splits`. Mô hình và ngưỡng dự đoán được lựa chọn trên tập xác thực; tập kiểm thử chỉ được dùng để báo cáo kết quả cuối cùng.

## Cấu hình môi trường và đường dẫn

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Hiển thị bảng kết quả đầy đủ để thuận tiện kiểm tra trong notebook.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Hỗ trợ chạy notebook từ thư mục notebooks hoặc từ thư mục gốc của kho mã.
root_candidates = [Path(".."), Path(".")]
PROJECT_ROOT = next(
    (path.resolve() for path in root_candidates if (path / "outputs" / "splits").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy outputs/splits. Hãy chạy notebook chuẩn bị dữ liệu trước."
    )

SPLIT_DIR = PROJECT_ROOT / "outputs" / "splits"
PREDICTION_DIR = PROJECT_ROOT / "outputs" / "predictions"
MODEL_DIR = PROJECT_ROOT / "outputs" / "models"
IMPORTANCE_DIR = PROJECT_ROOT / "outputs" / "feature_importance"

# Tạo các thư mục đầu ra nếu chúng chưa tồn tại.
for output_dir in [PREDICTION_DIR, MODEL_DIR, IMPORTANCE_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

# Cố định hạt giống ngẫu nhiên để kết quả có thể tái lập.
RANDOM_STATE = 42

print("Thư mục dự án:", PROJECT_ROOT)
print("Thư mục dữ liệu đã chia:", SPLIT_DIR)

## Nạp các tập dữ liệu đã được chia cố định

Notebook không tự chia lại dữ liệu. Mỗi file được kiểm tra với bảng ánh xạ `row_id → split` để bảo đảm Machine Learning và Rule-based Scoring sử dụng cùng các sinh viên trong từng tập.

In [ ]:
def load_fixed_splits(file_prefix):
    """Đọc ba tập dữ liệu và kiểm tra chúng với file ánh xạ chính thức."""

    # Đọc file ánh xạ được tạo trong notebook chuẩn bị dữ liệu.
    split_map_path = SPLIT_DIR / f"{file_prefix}_split.csv"
    split_map = pd.read_csv(split_map_path)
    assert split_map["row_id"].is_unique, f"{file_prefix}: row_id bị trùng."

    frames = {}
    for split_name in ["train", "validation", "test"]:
        # Đọc dữ liệu đầy đủ của từng tập và kiểm tra row_id tương ứng.
        frame_path = SPLIT_DIR / f"{file_prefix}_{split_name}.csv"
        frame = pd.read_csv(frame_path)
        expected_ids = set(
            split_map.loc[split_map["split"] == split_name, "row_id"]
        )
        assert frame["row_id"].is_unique, (
            f"{file_prefix}/{split_name}: row_id bị trùng."
        )
        assert set(frame["row_id"]) == expected_ids, (
            f"{file_prefix}/{split_name}: row_id không khớp file ánh xạ."
        )
        frames[split_name] = frame

    return frames


# Hai nguồn được nạp và lưu trong hai cấu trúc độc lập.
source_1_frames = load_fixed_splits("student_dropout_and_success")
source_2_frames = load_fixed_splits("student_dropout")

for source_name, frames in {
    "student_dropout_and_success": source_1_frames,
    "student_dropout": source_2_frames,
}.items():
    sizes = {split_name: len(frame) for split_name, frame in frames.items()}
    print(source_name, sizes)

## Xác định đặc trưng cho từng nguồn

Các cột định danh và biến mục tiêu không được đưa vào mô hình. Đối với Nguồn 1, các cột mã hóa danh mục được khai báo rõ ràng thay vì coi các mã số này là đại lượng liên tục.

In [ ]:
# Danh sách các cột dạng danh mục của Nguồn 1 theo mô tả bộ dữ liệu.
SOURCE_1_CATEGORICAL_COLUMNS = [
    "Marital status",
    "Application mode",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
]

# Nguồn 2 có các cột phân loại ở dạng chuỗi nên có thể nhận diện theo kiểu dữ liệu.
SOURCE_2_CATEGORICAL_COLUMNS = (
    source_2_frames["train"]
    .select_dtypes(include=["object", "str", "category"])
    .columns.tolist()
)

SOURCE_CONFIGS = {
    "student_dropout_and_success": {
        "frames": source_1_frames,
        "drop_columns": ["row_id", "Target", "Dropout"],
        "categorical_columns": SOURCE_1_CATEGORICAL_COLUMNS,
    },
    "student_dropout": {
        "frames": source_2_frames,
        "drop_columns": ["row_id", "Student_ID", "Dropout"],
        "categorical_columns": SOURCE_2_CATEGORICAL_COLUMNS,
    },
}

# Kiểm tra biến mục tiêu và các cột phân loại trước khi huấn luyện.
for source_name, config in SOURCE_CONFIGS.items():
    train_frame = config["frames"]["train"]
    assert set(train_frame["Dropout"].unique()) == {0, 1}
    missing_categorical = set(config["categorical_columns"]) - set(train_frame.columns)
    assert not missing_categorical, (
        f"{source_name}: thiếu cột phân loại {missing_categorical}"
    )
    feature_columns = [
        column for column in train_frame.columns if column not in config["drop_columns"]
    ]
    config["feature_columns"] = feature_columns
    print(
        f"{source_name}: {len(feature_columns)} đặc trưng, "
        f"{len(config['categorical_columns'])} cột phân loại"
    )

## Xây dựng pipeline tiền xử lý và mô hình

Mỗi pipeline tự thực hiện điền giá trị thiếu và mã hóa. Các bộ biến đổi chỉ được `fit` trên tập huấn luyện, nhờ đó thông tin từ tập xác thực và kiểm thử không rò rỉ vào mô hình.

In [ ]:
def build_preprocessor(feature_columns, categorical_columns, scale_numeric):
    """Tạo bộ tiền xử lý phù hợp với danh sách đặc trưng của một nguồn."""

    numeric_columns = [
        column for column in feature_columns if column not in categorical_columns
    ]

    # Cột số được điền bằng trung vị; Logistic Regression cần chuẩn hóa thang đo.
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipeline = Pipeline(numeric_steps)

    # Cột phân loại được điền bằng giá trị phổ biến nhất rồi mã hóa One-Hot.
    categorical_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=True),
            ),
        ]
    )

    return ColumnTransformer(
        [
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_candidate_models(feature_columns, categorical_columns):
    """Khởi tạo hai mô hình ứng viên với cấu hình có thể tái lập."""

    # Logistic Regression là đường cơ sở dễ giải thích.
    logistic_pipeline = Pipeline(
        [
            (
                "preprocessor",
                build_preprocessor(
                    feature_columns, categorical_columns, scale_numeric=True
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    # LightGBM là mô hình nâng cao dành cho dữ liệu dạng bảng.
    lightgbm_pipeline = Pipeline(
        [
            (
                "preprocessor",
                build_preprocessor(
                    feature_columns, categorical_columns, scale_numeric=False
                ),
            ),
            (
                "classifier",
                LGBMClassifier(
                    objective="binary",
                    n_estimators=300,
                    learning_rate=0.05,
                    num_leaves=31,
                    max_depth=-1,
                    min_child_samples=20,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    deterministic=True,
                    force_col_wise=True,
                    n_jobs=-1,
                    verbosity=-1,
                ),
            ),
        ]
    )

    return {
        "logistic_regression": logistic_pipeline,
        "lightgbm": lightgbm_pipeline,
    }

## Quy tắc lựa chọn ngưỡng và đánh giá

Ngưỡng mặc định 0,5 không phải lúc nào cũng phù hợp với hệ thống cảnh báo sớm. Notebook tìm ngưỡng trên tập xác thực bằng F2-score, chỉ số ưu tiên Recall cao hơn Precision để giảm nguy cơ bỏ sót sinh viên có khả năng bỏ học.

In [ ]:
def select_threshold(y_true, probability):
    """Chọn ngưỡng có F2 cao nhất trên tập xác thực."""

    threshold_rows = []
    # Duyệt ngưỡng từ 0,05 đến 0,95 với khoảng cách 0,01.
    for threshold in np.round(np.arange(0.05, 0.951, 0.01), 2):
        prediction = (probability >= threshold).astype(int)
        threshold_rows.append(
            {
                "threshold": float(threshold),
                "precision": precision_score(
                    y_true, prediction, zero_division=0
                ),
                "recall": recall_score(y_true, prediction, zero_division=0),
                "f2": fbeta_score(
                    y_true, prediction, beta=2, zero_division=0
                ),
            }
        )

    threshold_table = pd.DataFrame(threshold_rows)
    # Nếu F2 bằng nhau, ưu tiên Recall rồi Precision cao hơn.
    best_row = threshold_table.sort_values(
        ["f2", "recall", "precision", "threshold"],
        ascending=[False, False, False, False],
    ).iloc[0]
    return float(best_row["threshold"]), threshold_table


def calculate_metrics(y_true, probability, threshold):
    """Tính đầy đủ các chỉ số bắt buộc của hợp đồng thí nghiệm."""

    prediction = (probability >= threshold).astype(int)
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        y_true, prediction, labels=[0, 1]
    ).ravel()

    return {
        "threshold": float(threshold),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "f2": fbeta_score(y_true, prediction, beta=2, zero_division=0),
        "accuracy": accuracy_score(y_true, prediction),
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc": average_precision_score(y_true, probability),
        "tn": int(true_negative),
        "fp": int(false_positive),
        "fn": int(false_negative),
        "tp": int(true_positive),
    }

## Huấn luyện và so sánh trên tập xác thực

Hai thuật toán được huấn luyện riêng trên từng nguồn. Mỗi mô hình trả về xác suất bỏ học bằng `predict_proba`; không sử dụng nhãn `predict` thay cho xác suất.

In [ ]:
fitted_models = {}
validation_rows = []
threshold_tables = {}

for source_name, config in SOURCE_CONFIGS.items():
    frames = config["frames"]
    feature_columns = config["feature_columns"]
    categorical_columns = config["categorical_columns"]

    # Tách đặc trưng và nhãn; tuyệt đối không đưa row_id hoặc cột mục tiêu vào X.
    X_train = frames["train"][feature_columns]
    y_train = frames["train"]["Dropout"].astype(int)
    X_validation = frames["validation"][feature_columns]
    y_validation = frames["validation"]["Dropout"].astype(int)

    candidates = build_candidate_models(feature_columns, categorical_columns)
    for model_name, pipeline in candidates.items():
        print(f"Đang huấn luyện {source_name} / {model_name} ...")

        # Pipeline chỉ học bộ điền thiếu, bộ mã hóa và mô hình từ tập huấn luyện.
        pipeline.fit(X_train, y_train)
        validation_probability = pipeline.predict_proba(X_validation)[:, 1]
        assert np.all((validation_probability >= 0) & (validation_probability <= 1))

        # Ngưỡng cảnh báo được chọn hoàn toàn trên tập xác thực.
        threshold, threshold_table = select_threshold(
            y_validation, validation_probability
        )
        metrics = calculate_metrics(
            y_validation, validation_probability, threshold
        )
        metrics.update(
            {
                "data_source": source_name,
                "solution_type": "ml",
                "model": model_name,
            }
        )

        fitted_models[(source_name, model_name)] = pipeline
        threshold_tables[(source_name, model_name)] = threshold_table
        validation_rows.append(metrics)

validation_comparison = pd.DataFrame(validation_rows)
validation_comparison = validation_comparison[
    [
        "data_source", "solution_type", "model", "threshold",
        "precision", "recall", "f1", "f2", "accuracy",
        "roc_auc", "pr_auc", "tn", "fp", "fn", "tp",
    ]
]

print("\nKết quả trên tập xác thực:")
print(validation_comparison.round(4).to_string(index=False))

## Lựa chọn mô hình cho từng nguồn

Mô hình có Recall cao nhất được ưu tiên để hạn chế bỏ sót sinh viên có nguy cơ. Thứ tự tiếp theo là F2-score, PR-AUC, F1 và Accuracy. Việc lựa chọn không sử dụng bất kỳ thông tin nào từ tập kiểm thử.

In [ ]:
# Sắp xếp kết quả theo đúng thứ tự ưu tiên của hợp đồng thí nghiệm.
selected_models = (
    validation_comparison.sort_values(
        ["data_source", "recall", "f2", "pr_auc", "f1", "accuracy"],
        ascending=[True, False, False, False, False, False],
    )
    .groupby("data_source", as_index=False)
    .first()
)

assert set(selected_models["data_source"]) == set(SOURCE_CONFIGS)
print(
    selected_models[["data_source", "model", "threshold", "recall", "f2"]]
    .round(4)
    .to_string(index=False)
)

## Đánh giá cuối cùng trên tập kiểm thử

Mỗi nguồn chỉ sử dụng mô hình và ngưỡng đã được chọn trên tập xác thực. Không điều chỉnh mô hình dựa trên kết quả kiểm thử.

In [ ]:
test_rows = []
prediction_outputs = {}
selected_model_objects = {}

for selected in selected_models.to_dict(orient="records"):
    source_name = selected["data_source"]
    model_name = selected["model"]
    threshold = float(selected["threshold"])
    config = SOURCE_CONFIGS[source_name]
    test_frame = config["frames"]["test"]

    # Lấy đúng mô hình đã fit trên train và được chọn bằng validation.
    pipeline = fitted_models[(source_name, model_name)]
    X_test = test_frame[config["feature_columns"]]
    y_test = test_frame["Dropout"].astype(int)
    test_probability = pipeline.predict_proba(X_test)[:, 1]
    test_prediction = (test_probability >= threshold).astype(int)

    # Tính metric cuối cùng nhưng không dùng kết quả này để thay đổi mô hình.
    metrics = calculate_metrics(y_test, test_probability, threshold)
    metrics.update(
        {
            "data_source": source_name,
            "solution_type": "ml",
            "model": model_name,
        }
    )
    test_rows.append(metrics)

    # File dự đoán giữ row_id để đối chiếu trực tiếp với Rule-based Scoring.
    prediction_outputs[source_name] = pd.DataFrame(
        {
            "row_id": test_frame["row_id"].to_numpy(),
            "y_true": y_test.to_numpy(),
            "prediction_score": test_probability,
            "y_pred": test_prediction,
            "model": model_name,
            "threshold": threshold,
        }
    ).sort_values("row_id")
    selected_model_objects[source_name] = pipeline

test_metrics = pd.DataFrame(test_rows)[
    [
        "data_source", "solution_type", "model", "threshold",
        "precision", "recall", "f1", "f2", "accuracy",
        "roc_auc", "pr_auc", "tn", "fp", "fn", "tp",
    ]
]

print("Kết quả cuối cùng trên tập kiểm thử:")
print(test_metrics.round(4).to_string(index=False))

## Trích xuất mức độ quan trọng của đặc trưng

Logistic Regression sử dụng trị tuyệt đối của hệ số; LightGBM sử dụng mức độ quan trọng do mô hình cung cấp. Kết quả giúp nhóm giải thích yếu tố nào ảnh hưởng nhiều đến cảnh báo, nhưng không được diễn giải như quan hệ nhân quả.

In [ ]:
feature_importance_outputs = {}

for source_name, pipeline in selected_model_objects.items():
    # Lấy tên đặc trưng sau khi điền thiếu và mã hóa One-Hot.
    feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
    classifier = pipeline.named_steps["classifier"]

    if hasattr(classifier, "coef_"):
        importance_values = np.abs(classifier.coef_[0])
        importance_method = "absolute_coefficient"
    else:
        importance_values = classifier.feature_importances_
        importance_method = "feature_importance"

    importance_frame = pd.DataFrame(
        {
            "feature": feature_names,
            "importance": importance_values,
            "method": importance_method,
        }
    ).sort_values("importance", ascending=False, ignore_index=True)
    feature_importance_outputs[source_name] = importance_frame

    print(f"\nMười đặc trưng nổi bật của {source_name}:")
    print(importance_frame.head(10).round(4).to_string(index=False))

## Lưu kết quả và mô hình

Các file dự đoán tuân theo hợp đồng `row_id,y_true,prediction_score,y_pred`. Bảng metric được lưu riêng cho tập xác thực và tập kiểm thử. File `solution_comparison.csv` chưa được tạo ở đây vì chỉ được lập sau khi Rule-based Scoring hoàn tất.

In [ ]:
# Lưu kết quả so sánh ứng viên và metric cuối cùng.
validation_comparison.to_csv(
    PROJECT_ROOT / "outputs" / "ml_validation_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)
test_metrics.to_csv(
    PROJECT_ROOT / "outputs" / "ml_test_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

model_metadata = {}
for source_name, prediction_frame in prediction_outputs.items():
    # Lưu xác suất thật từ predict_proba và nhãn theo ngưỡng đã chọn.
    prediction_path = PREDICTION_DIR / f"{source_name}_ml.csv"
    prediction_frame.to_csv(prediction_path, index=False, encoding="utf-8-sig")

    # Lưu pipeline hoàn chỉnh để backend có thể tái sử dụng đúng bộ tiền xử lý.
    model_path = MODEL_DIR / f"{source_name}_ml.joblib"
    joblib.dump(selected_model_objects[source_name], model_path)

    # Lưu mức độ quan trọng của toàn bộ đặc trưng đã biến đổi.
    importance_path = IMPORTANCE_DIR / f"{source_name}_ml_feature_importance.csv"
    feature_importance_outputs[source_name].to_csv(
        importance_path, index=False, encoding="utf-8-sig"
    )

    selected_row = test_metrics.loc[test_metrics["data_source"] == source_name].iloc[0]
    model_metadata[source_name] = {
        "model": selected_row["model"],
        "threshold": float(selected_row["threshold"]),
        "feature_columns": SOURCE_CONFIGS[source_name]["feature_columns"],
        "categorical_columns": SOURCE_CONFIGS[source_name]["categorical_columns"],
        "random_state": RANDOM_STATE,
    }

# File JSON giúp backend biết mô hình, ngưỡng và thứ tự đặc trưng của từng nguồn.
with (MODEL_DIR / "ml_model_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(model_metadata, file, ensure_ascii=False, indent=2)

print("Đã lưu kết quả Machine Learning vào thư mục outputs.")

## Kiểm tra đầu ra

Phần kiểm tra cuối bảo đảm mỗi file dự đoán có đúng các `row_id` thuộc tập kiểm thử, xác suất nằm trong khoảng 0–1 và số dòng khớp dữ liệu đầu vào.

In [ ]:
required_prediction_columns = {
    "row_id", "y_true", "prediction_score", "y_pred"
}

for source_name, prediction_frame in prediction_outputs.items():
    test_frame = SOURCE_CONFIGS[source_name]["frames"]["test"]
    # Kiểm tra cấu trúc, miền xác suất và sự tương ứng với tập kiểm thử.
    assert required_prediction_columns.issubset(prediction_frame.columns)
    assert prediction_frame["row_id"].is_unique
    assert set(prediction_frame["row_id"]) == set(test_frame["row_id"])
    assert prediction_frame["prediction_score"].between(0, 1).all()
    assert set(prediction_frame["y_true"].unique()).issubset({0, 1})
    assert set(prediction_frame["y_pred"].unique()).issubset({0, 1})

print("Kiểm tra hoàn tất: tất cả file dự đoán đều hợp lệ.")

## Kết luận

Hai nguồn đã có pipeline Machine Learning độc lập, cùng sử dụng cách chia dữ liệu cố định. Logistic Regression và LightGBM được so sánh trên tập xác thực; chỉ mô hình được chọn mới được đánh giá trên tập kiểm thử. Các kết quả này chưa đủ để kết luận Machine Learning tốt hơn Rule-based Scoring cho đến khi hai bộ luật được chạy trên chính các `row_id` kiểm thử tương ứng.